In [ ]:
import numpy as np
np.random.seed(42)

# 1000 products, 5 sensor readings each, values between 0 and 1
n_samples = 1000
X = np.random.rand(n_samples, 5)

# Simple rule: if weighted sum of sensors crosses a threshold -> Acceptable (1)
weights_true = np.array([0.6, -0.4, 0.5, -0.3, 0.4])
score = X.dot(weights_true)

# Add a little randomness so it's not a perfectly clean split
noise = np.random.normal(0, 0.1, n_samples)
score = score + noise

y = (score > score.mean()).astype(float).reshape(-1, 1)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Acceptable (1):", int(y.sum()), " Defective (0):", int(len(y) - y.sum()))
print("\nFirst 5 rows of X:\n", X[:5])
print("First 5 labels:\n", y[:5].flatten())

X shape: (1000, 5)
y shape: (1000, 1)
Acceptable (1): 491  Defective (0): 509

First 5 rows of X:
 [[0.37454012 0.95071431 0.73199394 0.59865848 0.15601864]
 [0.15599452 0.05808361 0.86617615 0.60111501 0.70807258]
 [0.02058449 0.96990985 0.83244264 0.21233911 0.18182497]
 [0.18340451 0.30424224 0.52475643 0.43194502 0.29122914]
 [0.61185289 0.13949386 0.29214465 0.36636184 0.45606998]]
First 5 labels:
 [0. 0. 0. 0. 1.]


In [ ]:
split = int(0.8 * n_samples)

X_train, y_train = X[:split], y[:split]
X_test, y_test = X[split:], y[split:]

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 800
Testing samples: 200


In [ ]:
np.random.seed(1)

n_input = 5      # 5 sensors
n_hidden = 4     # 4 hidden neurons
n_output = 1     # 1 output (Acceptable/Defective)

W1 = np.random.uniform(-1, 1, (n_input, n_hidden))   # weights: input -> hidden
b1 = np.random.uniform(-1, 1, (1, n_hidden))          # bias for hidden layer

W2 = np.random.uniform(-1, 1, (n_hidden, n_output))  # weights: hidden -> output
b2 = np.random.uniform(-1, 1, (1, n_output))          # bias for output layer

print("W1 (input->hidden):\n", W1)
print("b1:\n", b1)
print("W2 (hidden->output):\n", W2)
print("b2:\n", b2)

W1 (input->hidden):
 [[-0.16595599  0.44064899 -0.99977125 -0.39533485]
 [-0.70648822 -0.81532281 -0.62747958 -0.30887855]
 [-0.20646505  0.07763347 -0.16161097  0.370439  ]
 [-0.5910955   0.75623487 -0.94522481  0.34093502]
 [-0.1653904   0.11737966 -0.71922612 -0.60379702]]
b1:
 [[ 0.60148914  0.93652315 -0.37315164  0.38464523]]
W2 (hidden->output):
 [[ 0.7527783 ]
 [ 0.78921333]
 [-0.82991158]
 [-0.92189043]]
b2:
 [[-0.66033916]]


### **sigmoid activation function and forward propagation**

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Forward pass using the whole training set
z1 = X_train.dot(W1) + b1      # weighted sum going into hidden layer
a1 = sigmoid(z1)                # hidden layer activation

z2 = a1.dot(W2) + b2            # weighted sum going into output layer
a2 = sigmoid(z2)                # final predicted output (probability)

print("z1 shape:", z1.shape)
print("a1 shape:", a1.shape)
print("z2 shape:", z2.shape)
print("a2 shape:", a2.shape)
print("\nFirst 5 predictions (a2):\n", a2[:5])

z1 shape: (800, 4)
a1 shape: (800, 4)
z2 shape: (800, 1)
a2 shape: (800, 1)

First 5 predictions (a2):
 [[0.38389444]
 [0.4193348 ]
 [0.35805564]
 [0.40177245]
 [0.44312395]]


In [ ]:
error = y_train - a2
mse = np.mean(error ** 2)

print("First 5 errors:\n", error[:5])
print("\nInitial Mean Squared Error (MSE):", mse)

First 5 errors:
 [[-0.38389444]
 [-0.4193348 ]
 [-0.35805564]
 [-0.40177245]
 [ 0.55687605]]

Initial Mean Squared Error (MSE): 0.2445684860563643


### **backpropagation**

In [ ]:
def sigmoid_derivative(a):
    return a * (1 - a)

# Backpropagation (single step)
d_output = error * sigmoid_derivative(a2)          # how much output layer should change
d_hidden = d_output.dot(W2.T) * sigmoid_derivative(a1)  # how much hidden layer should change

print("d_output shape:", d_output.shape)
print("d_hidden shape:", d_hidden.shape)
print("\nFirst 5 d_output values:\n", d_output[:5])
print("\nFirst 5 d_hidden values:\n", d_hidden[:5])

d_output shape: (800, 1)
d_hidden shape: (800, 4)

First 5 d_output values:
 [[-0.09079852]
 [-0.10210514]
 [-0.08229973]
 [-0.09656655]
 [ 0.13741758]]

First 5 d_hidden values:
 [[-0.01533878 -0.01500628  0.00709448  0.02038691]
 [-0.01915036 -0.01154162  0.01044119  0.02260086]
 [-0.01484136 -0.01560126  0.01039226  0.01840527]
 [-0.0181592  -0.01376937  0.01233411  0.02145534]
 [ 0.02584567  0.01673563 -0.01394309 -0.03164489]]


In [ ]:
learning_rate = 0.1

W2 += learning_rate * a1.T.dot(d_output)
b2 += learning_rate * np.sum(d_output, axis=0, keepdims=True)

W1 += learning_rate * X_train.T.dot(d_hidden)
b1 += learning_rate * np.sum(d_hidden, axis=0, keepdims=True)

print("Updated W2:\n", W2)
print("Updated b2:\n", b2)

Updated W2:
 [[ 1.49218036]
 [ 2.05745282]
 [-0.72752631]
 [-0.30687566]]
Updated b2:
 [[0.79992268]]


### **initial and final loss values**

In [ ]:
np.random.seed(1)
W1 = np.random.uniform(-1, 1, (n_input, n_hidden))
b1 = np.random.uniform(-1, 1, (1, n_hidden))
W2 = np.random.uniform(-1, 1, (n_hidden, n_output))
b2 = np.random.uniform(-1, 1, (1, n_output))

learning_rate = 0.1
epochs = 5000
error_history = []

for epoch in range(epochs):
    # forward pass
    z1 = X_train.dot(W1) + b1
    a1 = sigmoid(z1)
    z2 = a1.dot(W2) + b2
    a2 = sigmoid(z2)

    # error
    error = y_train - a2
    mse = np.mean(error ** 2)
    error_history.append(mse)

    # backprop
    d_output = error * sigmoid_derivative(a2)
    d_hidden = d_output.dot(W2.T) * sigmoid_derivative(a1)

    # update weights
    W2 += learning_rate * a1.T.dot(d_output)
    b2 += learning_rate * np.sum(d_output, axis=0, keepdims=True)
    W1 += learning_rate * X_train.T.dot(d_hidden)
    b1 += learning_rate * np.sum(d_hidden, axis=0, keepdims=True)

    if epoch % 500 == 0:
        print(f"Epoch {epoch}, MSE: {mse:.6f}")

print(f"\nFinal MSE after {epochs} epochs: {error_history[-1]:.6f}")

Epoch 0, MSE: 0.244568
Epoch 500, MSE: 0.076525
Epoch 1000, MSE: 0.072500
Epoch 1500, MSE: 0.071252
Epoch 2000, MSE: 0.071123
Epoch 2500, MSE: 0.071138
Epoch 3000, MSE: 0.070822
Epoch 3500, MSE: 0.070443
Epoch 4000, MSE: 0.070074
Epoch 4500, MSE: 0.069739

Final MSE after 5000 epochs: 0.069547


In [ ]:
def predict(X, W1, b1, W2, b2):
    a1 = sigmoid(X.dot(W1) + b1)
    a2 = sigmoid(a1.dot(W2) + b2)
    return (a2 >= 0.5).astype(int)

train_preds = predict(X_train, W1, b1, W2, b2)
test_preds = predict(X_test, W1, b1, W2, b2)

train_acc = np.mean(train_preds == y_train)
test_acc = np.mean(test_preds == y_test)

print("Train Accuracy:", train_acc * 100, "%")
print("Test Accuracy:", test_acc * 100, "%")

print("\nSample predictions (first 10 test rows):")
for i in range(10):
    print("Actual:", int(y_test[i][0]), " Predicted:", int(test_preds[i][0]))

Train Accuracy: 89.625 %
Test Accuracy: 89.0 %

Sample predictions (first 10 test rows):
Actual: 1  Predicted: 0
Actual: 0  Predicted: 0
Actual: 1  Predicted: 1
Actual: 0  Predicted: 0
Actual: 1  Predicted: 1
Actual: 0  Predicted: 0
Actual: 0  Predicted: 0
Actual: 1  Predicted: 1
Actual: 1  Predicted: 1
Actual: 0  Predicted: 0


### **compare accuracy at 100, 1000, and 5000 epochs**

In [ ]:
def train_network(epochs, lr=0.1):
    np.random.seed(1)
    W1 = np.random.uniform(-1, 1, (n_input, n_hidden))
    b1 = np.random.uniform(-1, 1, (1, n_hidden))
    W2 = np.random.uniform(-1, 1, (n_hidden, n_output))
    b2 = np.random.uniform(-1, 1, (1, n_output))

    for epoch in range(epochs):
        a1 = sigmoid(X_train.dot(W1) + b1)
        a2 = sigmoid(a1.dot(W2) + b2)
        error = y_train - a2
        d_output = error * sigmoid_derivative(a2)
        d_hidden = d_output.dot(W2.T) * sigmoid_derivative(a1)
        W2 += lr * a1.T.dot(d_output)
        b2 += lr * np.sum(d_output, axis=0, keepdims=True)
        W1 += lr * X_train.T.dot(d_hidden)
        b1 += lr * np.sum(d_hidden, axis=0, keepdims=True)

    final_mse = np.mean((y_train - a2) ** 2)
    test_preds = predict(X_test, W1, b1, W2, b2)
    test_acc = np.mean(test_preds == y_test)
    return final_mse, test_acc

for ep in [100, 1000, 5000]:
    mse, acc = train_network(ep)
    print(f"Epochs: {ep} | Final MSE: {mse:.4f} | Test Accuracy: {acc*100:.2f}%")

Epochs: 100 | Final MSE: 0.1088 | Test Accuracy: 88.00%
Epochs: 1000 | Final MSE: 0.0728 | Test Accuracy: 89.50%
Epochs: 5000 | Final MSE: 0.0695 | Test Accuracy: 89.00%


### **compare learning rate 0.1 vs 1.0**

In [ ]:
for lr in [0.1, 1.0]:
    mse, acc = train_network(epochs=2000, lr=lr)
    print(f"Learning Rate: {lr} | Final MSE: {mse:.4f} | Test Accuracy: {acc*100:.2f}%")

Learning Rate: 0.1 | Final MSE: 0.0713 | Test Accuracy: 90.00%
Learning Rate: 1.0 | Final MSE: 0.5062 | Test Accuracy: 48.00%


### **does a bigger hidden layer (10 neurons instead of 4) always perform better?**

In [ ]:
def train_network_hidden(n_hidden_neurons, epochs=5000, lr=0.1):
    np.random.seed(1)
    W1 = np.random.uniform(-1, 1, (n_input, n_hidden_neurons))
    b1 = np.random.uniform(-1, 1, (1, n_hidden_neurons))
    W2 = np.random.uniform(-1, 1, (n_hidden_neurons, n_output))
    b2 = np.random.uniform(-1, 1, (1, n_output))

    for epoch in range(epochs):
        a1 = sigmoid(X_train.dot(W1) + b1)
        a2 = sigmoid(a1.dot(W2) + b2)
        error = y_train - a2
        d_output = error * sigmoid_derivative(a2)
        d_hidden = d_output.dot(W2.T) * sigmoid_derivative(a1)
        W2 += lr * a1.T.dot(d_output)
        b2 += lr * np.sum(d_output, axis=0, keepdims=True)
        W1 += lr * X_train.T.dot(d_hidden)
        b1 += lr * np.sum(d_hidden, axis=0, keepdims=True)

    final_mse = np.mean((y_train - a2) ** 2)
    test_preds = predict(X_test, W1, b1, W2, b2)
    test_acc = np.mean(test_preds == y_test)
    return final_mse, test_acc

for h in [4, 10]:
    mse, acc = train_network_hidden(h)
    print(f"Hidden Neurons: {h} | Final MSE: {mse:.4f} | Test Accuracy: {acc*100:.2f}%")

Hidden Neurons: 4 | Final MSE: 0.0695 | Test Accuracy: 89.00%
Hidden Neurons: 10 | Final MSE: 0.0665 | Test Accuracy: 89.50%


In [ ]:
def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

def train_relu(epochs=5000, lr=0.1):
    np.random.seed(1)
    W1 = np.random.uniform(-1, 1, (n_input, n_hidden))
    b1 = np.random.uniform(-1, 1, (1, n_hidden))
    W2 = np.random.uniform(-1, 1, (n_hidden, n_output))
    b2 = np.random.uniform(-1, 1, (1, n_output))

    mse_history = []
    for epoch in range(epochs):
        z1 = X_train.dot(W1) + b1
        a1 = relu(z1)                      # ReLU instead of sigmoid in hidden layer
        z2 = a1.dot(W2) + b2
        a2 = sigmoid(z2)                   # output layer stays sigmoid

        error = y_train - a2
        mse_history.append(np.mean(error ** 2))

        d_output = error * sigmoid_derivative(a2)
        d_hidden = d_output.dot(W2.T) * relu_derivative(z1)   # note: uses z1, not a1

        W2 += lr * a1.T.dot(d_output)
        b2 += lr * np.sum(d_output, axis=0, keepdims=True)
        W1 += lr * X_train.T.dot(d_hidden)
        b1 += lr * np.sum(d_hidden, axis=0, keepdims=True)

    test_preds = (sigmoid(relu(X_test.dot(W1)+b1).dot(W2)+b2) >= 0.5).astype(int)
    test_acc = np.mean(test_preds == y_test)
    return mse_history, test_acc

relu_history, relu_acc = train_relu()
print("Final MSE (ReLU):", relu_history[-1])
print("Test Accuracy (ReLU):", relu_acc * 100, "%")
print("\nMSE at checkpoints:")
for e in [0, 100, 500, 1000, 5000]:
    print(f"Epoch {e}: {relu_history[e-1] if e>0 else relu_history[0]:.4f}")

Final MSE (ReLU): 0.50625
Test Accuracy (ReLU): 48.0 %

MSE at checkpoints:
Epoch 0: 0.2290
Epoch 100: 0.5062
Epoch 500: 0.5062
Epoch 1000: 0.5062
Epoch 5000: 0.5062


In [ ]:
relu_history2, relu_acc2 = train_relu(epochs=5000, lr=0.01)
print("Final MSE (ReLU, lr=0.01):", relu_history2[-1])
print("Test Accuracy (ReLU, lr=0.01):", relu_acc2 * 100, "%")
print("\nMSE at checkpoints:")
for e in [0, 100, 500, 1000, 5000]:
    print(f"Epoch {e}: {relu_history2[e-1] if e>0 else relu_history2[0]:.4f}")

Final MSE (ReLU, lr=0.01): 0.06971355660611887
Test Accuracy (ReLU, lr=0.01): 89.5 %

MSE at checkpoints:
Epoch 0: 0.2290
Epoch 100: 0.0800
Epoch 500: 0.0709
Epoch 1000: 0.0705
Epoch 5000: 0.0697
